In [1]:
from sklearn.datasets import load_sample_image
import numpy as np
import torch
from torch.nn import functional as F
from torch import nn
import matplotlib.pyplot as plt

In [2]:
china = load_sample_image("china.jpg") / 255
flower = load_sample_image("flower.jpg") / 255

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(china)
axes[0].set_title("Imagem: China")
axes[1].imshow(flower)
axes[1].set_title("Imagem: Flor")
for ax in axes:
    ax.axis("off")
plt.show()

In [4]:
images = np.array([china, flower])
batch_size, height, width, channels = images.shape

filters = np.zeros(shape=(7, 7, channels, 2), dtype=np.float32)

filters[:, 3, :, 0] = 1
filters[3, :, :, 1] = 1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(filters[:, :, :, 0])
axes[0].set_title("Filtro 1 (vertical)")
axes[1].imshow(filters[:, :, :, 1])
axes[1].set_title("Filtro 2 (horizontal)")
for ax in axes:
    ax.axis("off")
plt.show()

In [6]:
conv = nn.Conv2d(in_channels=channels,
                    out_channels=2,
                    kernel_size=7,
                    bias=False)

# Transponha o tensor das imagens para ter forma (tamanho do lote, canais, altura, largura)
images_transposed = np.transpose(images, (0, 3, 1, 2))

outputs = conv(torch.from_numpy(images_transposed).float())

In [ ]:
output = outputs.detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(output[0, 0, :, :], cmap="gray")
axes[0].set_title("Saída do Filtro 1")
axes[1].imshow(output[0, 1, :, :], cmap="gray")
axes[1].set_title("Saída do Filtro 2")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(images[0])
axes[0].set_title("Original")
axes[1].imshow(output[0, 0, :, :], cmap="gray")
axes[1].set_title("Filtro 1 (detecção vertical)")
axes[2].imshow(output[0, 1, :, :], cmap="gray")
axes[2].set_title("Filtro 2 (detecção horizontal)")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.imshow(images[0])
plt.imshow(output[0, 0, :, :], cmap="gray", alpha=0.5)
plt.title("Filtro 1 sobreposto à imagem original")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.imshow(images[0])
plt.imshow(output[0, 1, :, :], cmap="gray", alpha=0.5)
plt.title("Filtro 2 sobreposto à imagem original")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(16, 8))
plt.imshow(outputs[0, 1, :, :].detach().numpy(), cmap="gray")
plt.show()

In [12]:
max_pool = nn.MaxPool2d(kernel_size=2)
output_max_pool = max_pool(outputs)

In [13]:
global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
output_global_avg_pool = global_avg_pool(outputs)

In [14]:
class CNN(nn.Module):
    def __init__(self, input_channels=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=input_channels, out_channels=4, kernel_size=3, padding=3)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, padding=2)
        self.pool2 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(in_channels=8, out_channels=8, kernel_size=3, padding=1)

    def forward(self, x):
        x1 = F.relu(self.conv1(x))
        x2 = self.pool1(x1)
        x3 = F.relu(self.conv2(x2))
        x4 = self.pool2(x3)
        x5 = F.relu(self.conv3(x4))
        return [x1, x2, x3, x4, x5]  # retorna as ativações intermediárias

In [15]:
def visualize_activations(model, x):
    activations = []
    hooks = []

    def hook_fn(module, input, output):
        activations.append(output.detach())

    # registra hooks nas camadas convolucionais
    for layer in [model.conv1, model.conv2, model.conv3]:
        hooks.append(layer.register_forward_hook(hook_fn))

    # faz uma passada
    _ = model(x)

    # remove os hooks
    for h in hooks:
        h.remove()

    # plota os mapas de ativação
    import matplotlib.pyplot as plt
    for i, act in enumerate(activations):
        n_filters = act.shape[1]
        fig, axes = plt.subplots(1, min(n_filters, 4), figsize=(12, 4))
        for j in range(min(n_filters, 4)):
            axes[j].imshow(act[0, j, :, :], cmap="gray")
            axes[j].set_title(f"Conv{i+1} - Filtro {j+1}")
            axes[j].axis("off")
        plt.show()

In [16]:
images = np.array([china, flower])
batch_size, height, width, channels = images.shape
images_torch = torch.from_numpy(np.transpose(images, (0, 3, 1, 2))).float()

model = CNN(input_channels=channels)
activations = model(images_torch)

In [ ]:
def plot_activations(activations, titles):
    for i, act in enumerate(activations):
        n_filters = act.shape[1]
        fig, axes = plt.subplots(1, min(n_filters, 4), figsize=(12, 4))
        fig.suptitle(titles[i], fontsize=14)
        for j in range(min(n_filters, 4)):
            axes[j].imshow(act[0, j, :, :].detach().numpy(), cmap="gray")
            axes[j].axis("off")
        plt.show()

titles = [
    "Saída Conv1 (detecção de padrões iniciais)",
    "Saída após Pool1 (redução de dimensão)",
    "Saída Conv2 (padrões intermediários)",
    "Saída após Pool2 (resumo das features)",
    "Saída Conv3 (detalhes refinados)"
]

plot_activations(activations, titles)

In [18]:
class LeNet5(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
    self.avg_pool1 = nn.AvgPool2d(kernel_size=2, stride=2)

    self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
    self.avg_pool2 = nn.AvgPool2d(kernel_size=2, stride=2)

    self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5)

    self.flatten = nn.Flatten()

    self.fc1 = nn.Linear(120, 84)
    self.fc2 = nn.Linear(84, 10)

  def forward(self, x):
      x = F.tanh(self.conv1(x))
      x = self.avg_pool1(x)
      print("Shape apos Pool 1:", x.shape)

      x = F.tanh(self.conv2(x))
      x = self.avg_pool2(x)
      print("Shape apos Pool 2:", x.shape)

      x = F.tanh(self.conv3(x))
      print("Shape apos conv3:", x.shape)

      x = self.flatten(x)
      print("Shape apos flatten:", x.shape)
      x = F.tanh(self.fc1(x))

      return self.fc2(x)

In [19]:
input = torch.randn(1, 1, 32, 32)
model = LeNet5()
output = model(input)
output.shape

Shape apos Pool 1: torch.Size([1, 6, 14, 14])
Shape apos Pool 2: torch.Size([1, 16, 5, 5])
Shape apos conv3: torch.Size([1, 120, 1, 1])
Shape apos flatten: torch.Size([1, 120])


torch.Size([1, 10])

In [20]:
class AlexNet(nn.Module):
  def __init__(self):
    super().__init__()

    self.ln = nn.Conv2d(in_channels=3, out_channels=96, kernel_size=11)

    self.c1 = nn.Conv2d(96, 256, kernel_size=11, stride=4, padding=0)
    self.s2 = nn.MaxPool2d(kernel_size=3, stride=2)

    self.c3 = nn.Conv2d(256, 384, kernel_size=5, stride=1, padding=2)
    self.s4 = nn.MaxPool2d(kernel_size=3, stride=1)

    self.c5 = nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1)
    self.c6 = nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1)
    self.c7 = nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1)

    self.flatten = nn.Flatten()

    self.f8 = nn.Linear(128*23*23, 64)
    self.f9 = nn.Linear(64, 32)

    self.out = nn.Linear(32, 10)

  def forward(self, x):
      x = F.relu(self.ln(x))
      print("Apos ln:", x.shape)

      x = F.relu(self.c1(x))
      x = self.s2(x)
      print("Apos s2:", x.shape)

      x = F.relu(self.c3(x))
      x = self.s4(x)
      print("Apos s4:", x.shape)

      x = F.relu(self.c5(x))
      x = F.relu(self.c6(x))
      x = F.relu(self.c7(x))

      print("Apos c7:", x.shape)

      x = self.flatten(x)
      print("Apos flatten:", x.shape)

      x = F.relu(self.f8(x))
      x = F.relu(self.f9(x))

      return self.out(x)

In [21]:
def calcule_out_conv(input_size, kernel_size, stride_size, padding_size):
    return int((input_size - kernel_size + 2 * padding_size) / stride_size + 1)

In [22]:
def calculate_out_pool(input_size, kernel_size, stride_size):
    return int((input_size - kernel_size) / stride_size + 1)

In [23]:
out_ln = calcule_out_conv(224, 11, 1, 0)
print("Após ln:", out_ln)

out_c1 = calcule_out_conv(out_ln, 11, 4, 0)
out_s2 = calculate_out_pool(out_c1, 3, 2)
print("Após S2:", out_s2)

out_c2 = calcule_out_conv(out_s2, 5, 1, 2)
out_s4 = calculate_out_pool(out_c2, 3, 1)
print("Após s4:", out_s4)

out_c5 = calcule_out_conv(out_s4, 3, 1, 1)
out_c6 = calcule_out_conv(out_c5, 3, 1, 1)
out_c7 = calcule_out_conv(out_c6, 3, 1, 1)
print("Após c7:", out_c7)

Após ln: 214
Após S2: 25
Após s4: 23
Após c7: 23


In [24]:
model = AlexNet()

input = torch.randn(1, 3, 224, 224)
output = model(input)
print(output.shape)

Apos ln: torch.Size([1, 96, 214, 214])
Apos s2: torch.Size([1, 256, 25, 25])
Apos s4: torch.Size([1, 384, 23, 23])
Apos c7: torch.Size([1, 128, 23, 23])
Apos flatten: torch.Size([1, 67712])
torch.Size([1, 10])


In [25]:
class SeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=1):
        super().__init__()

        # Depthwise: aplica um filtro por canal
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,  # 👉 chave: separa os canais
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)

        # Pointwise: mistura os canais com 1x1
        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = F.relu(self.bn1(self.depthwise(x)))
        x = F.relu(self.bn2(self.pointwise(x)))
        return x

In [26]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # Caminho principal (2 convoluções separáveis)
        self.conv1 = SeparableConv(in_channels, out_channels, kernel_size=3, stride=stride)
        self.conv2 = SeparableConv(out_channels, out_channels, kernel_size=3, stride=1)

        # Caminho de atalho (skip connection)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out += self.shortcut(x)
        return F.relu(out)

In [27]:
class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Primeira camada convolucional: transforma imagem RGB (3 canais) em 16 mapas de ativação
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)

        # Normaliza a saída da primeira conv pra estabilizar o treino e acelerar a convergência
        self.bn1 = nn.BatchNorm2d(16)

        # === BLOCO RESIDUAL 1 ===
        # Mantém o mesmo número de canais (16→16) e a mesma resolução espacial
        # Esse bloco aprende correções sutis na representação inicial
        self.layer1 = ResidualBlock(16, 16)

        # === BLOCO RESIDUAL 2 ===
        # Dobra o número de canais (16→32) e reduz pela metade a resolução (stride=2)
        # Começa a extrair padrões mais abstratos e compactos
        self.layer2 = ResidualBlock(16, 32, stride=2)

        # === BLOCO RESIDUAL 3 ===
        # Novamente dobra os canais (32→64) e reduz a resolução
        # Aqui a rede aprende representações de alto nível
        self.layer3 = ResidualBlock(32, 64, stride=2)

        # Camada de pooling adaptativo que força a saída a ter dimensão espacial 1x1,
        # independentemente do tamanho da entrada
        # Ou seja, cada canal vira um único valor médio
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Camada totalmente conectada que transforma o vetor de 64 features em 10 classes (por padrão)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # 1️ Primeira conv + BN + ReLU (extração inicial de padrões locais)
        out = F.relu(self.bn1(self.conv1(x)))

        # 2️ Passa pelos blocos residuais (aprendem correções incrementais)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)

        # 3️ Pooling global: resume cada canal em um único valor (espacialmente).
        out = self.avgpool(out)

        # 4️ Achata (de [batch, canais, 1, 1] → [batch, canais])
        out = torch.flatten(out, 1)

        # 5️ Classificador final (Linear)
        out = self.fc(out)

        return out

In [28]:
x = torch.randn(1, 3, 64, 64)
model = ResNet()
y = model(x)
print(y.shape)

torch.Size([1, 10])


In [29]:
from torchvision.models import resnet50, ResNet50_Weights

In [30]:
#  Novos pesos com precisão de 80,858%
base_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
base_model.fc = nn.Identity()

for name, param in base_model.named_parameters():
    if "layer4" not in name:
        param.requires_grad = False

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 144MB/s]


In [31]:
class Model(nn.Module):
    def __init__(self, base_model, num_classes=10):
        super().__init__()
        self.base = base_model
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.base(x)
        x = self.classifier(x)
        return x

In [32]:
model = Model(base_model)
for name, param in model.named_parameters():
    print(name, param.requires_grad)

base.conv1.weight False
base.bn1.weight False
base.bn1.bias False
base.layer1.0.conv1.weight False
base.layer1.0.bn1.weight False
base.layer1.0.bn1.bias False
base.layer1.0.conv2.weight False
base.layer1.0.bn2.weight False
base.layer1.0.bn2.bias False
base.layer1.0.conv3.weight False
base.layer1.0.bn3.weight False
base.layer1.0.bn3.bias False
base.layer1.0.downsample.0.weight False
base.layer1.0.downsample.1.weight False
base.layer1.0.downsample.1.bias False
base.layer1.1.conv1.weight False
base.layer1.1.bn1.weight False
base.layer1.1.bn1.bias False
base.layer1.1.conv2.weight False
base.layer1.1.bn2.weight False
base.layer1.1.bn2.bias False
base.layer1.1.conv3.weight False
base.layer1.1.bn3.weight False
base.layer1.1.bn3.bias False
base.layer1.2.conv1.weight False
base.layer1.2.bn1.weight False
base.layer1.2.bn1.bias False
base.layer1.2.conv2.weight False
base.layer1.2.bn2.weight False
base.layer1.2.bn2.bias False
base.layer1.2.conv3.weight False
base.layer1.2.bn3.weight False
base.lay

In [33]:
import timm
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

In [34]:
base_model = timm.create_model('xception', pretrained=True)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth


In [35]:
class Model(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        self.loc = nn.Linear(1000, 4)

    def forward(self, x):
        x = self.base(x)
        return self.loc(x)

In [36]:
model = Model(base_model)
config = resolve_data_config({}, model=model)
transform = create_transform(**config)

img = Image.fromarray((images[0] * 255).astype(np.uint8)).convert('RGB')
tensor = transform(img).unsqueeze(0)

with torch.no_grad():
    out = model(tensor)
    probabilities = torch.nn.functional.softmax(out[0], dim=0)

In [37]:
out, probabilities

(tensor([[-0.0249,  0.0761, -0.0136,  0.0237]]),
 tensor([0.2400, 0.2655, 0.2427, 0.2519]))

In [38]:
from torchvision.ops import box_iou

ground_truth_box = torch.tensor([-0.0052, -0.910, 0.009, -0.0461])
ious = box_iou(out, ground_truth_box.unsqueeze(0))

In [39]:
ious.detach().numpy()

array([[0.]], dtype=float32)